# Localization Analysis — Unitree go2

**Course:** Ciência de Dados — FEI Mestrado  
**Description:** Analysis of RTABMAP localization quality using one or two cameras in a go2 robot.

---

### Data Sources
| File | Description |
|------|-------------|
| `localization_log.csv` | Per-update metrics from `/rtabmap/info` and `/localization_pose` (inliers, covariance, pose, etc.) |
| `plan_log.csv` | Planned path poses from `/plan` topic, grouped by `plan_id` |

### Sections
1. **Data Loading and Data Processing** — read CSV logs from a given run folder and process all data 
2. **Path Visualization** — planned path vs actual robot trajectory  
3. **Localization Quality** — inliers, hypothesis ratio, covariance over time


### 1. **Data Loading and Data Processing**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Read the .csv logs from localization and planning navigation
loc_1  = pd.read_csv('localization_analysis/data/logs/logger_csv_1/localization_log.csv')
loc_2  = pd.read_csv('localization_analysis/data/logs/logger_csv_2/localization_log.csv')
plan = pd.read_csv('localization_analysis/data/logs/logger_csv_1/plan_log.csv')

In [18]:
# Take the first path that NAV2 stack calculated (the plan_id = 1)
last_plan = plan[plan['plan_id'] == plan['plan_id'].min()]
last_plan

,plan_id,pose_index,x,y
0,1,0,-2.7519,2.1549
1,1,1,-2.7019,2.1049
2,1,2,-2.6519,2.0549
3,1,3,-2.6019,2.0049
4,1,4,-2.5519,1.9549
...,...,...,...,...
159,1,159,1.7122,2.1049
160,1,160,1.7187,2.1549
161,1,161,1.7270,2.2049
162,1,162,1.7371,2.2549


In [19]:
# Drop rows where pose was not yet received (NaN)
actual = loc_1.dropna(subset=['pos_x', 'pos_y'])
actual

,timestamp_sec,camera_mode,node_id,inliers,matches,inlier_ratio,hypothesis_ratio,loop_closure_id,wm_size,detection_time_ms,total_time_ms,pos_x,pos_y,yaw,cov_xx,cov_yy,cov_yaw,cov_pos_trace
0,1.774646e+09,single,19532,4,35,0.0090,1.0,0,977,200.94,212.78,-3.0953,2.4795,2.3537,9999.000000,9999.000000,9999.000000,19998.000000
1,1.774646e+09,single,19533,3,31,0.0068,1.0,0,977,179.96,191.82,-3.0952,2.4796,2.3536,9999.000000,9999.000000,9999.000000,19998.000000
2,1.774646e+09,single,19534,0,0,0.0000,0.0,0,977,180.05,190.29,-3.1194,2.4870,2.3501,9999.000000,9999.000000,9999.000000,19998.000000
3,1.774646e+09,single,19535,0,0,0.0000,0.0,0,977,156.80,161.43,-3.1141,2.4825,2.3498,9999.000000,9999.000000,9999.000000,19998.000000
4,1.774646e+09,single,19536,0,0,0.0000,0.0,0,977,154.73,161.19,-3.1174,2.4805,2.3486,9999.000000,9999.000000,9999.000000,19998.000000
5,1.774646e+09,single,19537,3,22,0.0073,1.0,0,977,208.54,226.50,-2.9903,2.3139,2.4410,9999.000000,9999.000000,9999.000000,19998.000000
6,1.774646e+09,single,19538,0,0,0.0000,0.0,0,977,228.65,237.78,-2.7998,2.1400,2.4517,9999.000000,9999.000000,9999.000000,19998.000000
7,1.774646e+09,single,19539,3,23,0.0066,1.0,0,977,251.72,261.95,-2.7540,2.1281,2.4599,9999.000000,9999.000000,9999.000000,19998.000000
8,1.774646e+09,single,19540,3,34,0.0064,1.0,0,977,193.76,200.83,-2.7563,2.1308,2.4618,9999.000000,9999.000000,9999.000000,19998.000000
9,1.774646e+09,single,19541,0,0,0.0000,0.0,0,977,243.48,252.27,-2.7583,2.1327,2.4630,9999.000000,9999.000000,9999.000000,19998.000000


In [20]:
# Drop rows where pose was not yet received (NaN)
actual_2 = loc_2.dropna(subset=['pos_x', 'pos_y'])
actual_2

,timestamp_sec,camera_mode,node_id,inliers,matches,inlier_ratio,hypothesis_ratio,loop_closure_id,wm_size,detection_time_ms,total_time_ms,pos_x,pos_y,yaw,cov_xx,cov_yy,cov_yaw,cov_pos_trace
0,1.774649e+09,single,19732,4,34,0.0083,1.0,0,977,198.38,180.92,-3.1091,2.4706,2.3818,9999.000000,9999.000000,9999.000000,19998.000000
1,1.774649e+09,single,19735,0,0,0.0000,0.0,0,977,169.07,143.46,-3.1334,2.5126,2.3597,9999.000000,9999.000000,9999.000000,19998.000000
2,1.774649e+09,single,19737,3,24,0.0092,1.0,0,977,206.74,219.84,-3.0878,2.3817,2.4009,9999.000000,9999.000000,9999.000000,19998.000000
3,1.774649e+09,single,19740,3,35,0.0055,1.0,0,977,212.22,171.84,-2.7638,2.0170,2.4405,9999.000000,9999.000000,9999.000000,19998.000000
4,1.774649e+09,single,19746,4,37,0.0061,1.0,0,977,242.10,265.37,-2.6976,2.1347,2.5078,9999.000000,9999.000000,9999.000000,19998.000000
5,1.774649e+09,single,19749,0,0,0.0000,0.0,0,977,274.07,258.23,-2.0958,1.5141,-0.9542,9999.000000,9999.000000,9999.000000,19998.000000
6,1.774649e+09,single,19750,4,31,0.0114,1.0,0,977,227.07,189.80,-1.8762,1.2714,-0.9545,9999.000000,9999.000000,9999.000000,19998.000000
7,1.774649e+09,single,19751,3,33,0.0078,1.0,0,977,325.25,338.77,-1.6152,0.5843,-1.0585,9999.000000,9999.000000,9999.000000,19998.000000
8,1.774649e+09,single,19753,0,0,0.0000,0.0,0,977,318.58,414.04,-1.3235,-0.3290,-1.4142,9999.000000,9999.000000,9999.000000,19998.000000
9,1.774649e+09,single,19755,0,0,0.0000,0.0,0,977,303.42,350.93,-1.2579,-0.8513,-0.8249,9999.000000,9999.000000,9999.000000,19998.000000


### 2. **Path Visualization**

In [22]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Run 1', 'Run 2'],
                    shared_yaxes=True)

for col, (actual_df, title) in enumerate(zip([actual, actual_2], ['Run 1', 'Run 2']), start=1):
    fig.add_trace(go.Scatter(
        x=last_plan['x'], y=last_plan['y'],
        mode='lines', name='Planned path',
        line=dict(color='royalblue', dash='dash', width=2),
        legendgroup='plan', showlegend=(col == 1)
    ), row=1, col=col)
    fig.add_trace(go.Scatter(
        x=actual_df['pos_x'], y=actual_df['pos_y'],
        mode='lines', name=title,
        line=dict(color='tomato', width=2),
        legendgroup=title
    ), row=1, col=col)

fig.update_yaxes(scaleanchor='x', scaleratio=1)
fig.update_layout(title='Planned vs Actual Path', hovermode='closest')
fig.show()

#### 2.1 **Normalizing and get the median of all paths**

In [23]:
from scipy.interpolate import interp1d
import numpy as np

def normalize_path(pos_x, pos_y, n_points=500):
    """Resample a path to n_points using arc-length parameterization."""
    coords = np.column_stack([pos_x, pos_y])
    
    # Compute cumulative arc-length
    deltas = np.diff(coords, axis=0)
    seg_lengths = np.hypot(deltas[:, 0], deltas[:, 1])
    arc = np.concatenate([[0], np.cumsum(seg_lengths)])
    arc_norm = arc / arc[-1]  # normalize to [0, 1]
    
    # Interpolate x and y independently over the normalized arc
    fx = interp1d(arc_norm, pos_x, kind='linear')
    fy = interp1d(arc_norm, pos_y, kind='linear')
    
    t = np.linspace(0, 1, n_points)
    return fx(t), fy(t)

# Resample all paths to the same number of points
n_points = 500
paths = [actual, actual_2]  # add more here as you collect them

resampled = [normalize_path(df['pos_x'].values, df['pos_y'].values, n_points) for df in paths]

xs = np.array([p[0] for p in resampled])  # shape: (n_paths, n_points)
ys = np.array([p[1] for p in resampled])

# Median path
median_x = np.median(xs, axis=0)
median_y = np.median(ys, axis=0)


In [24]:
fig = go.Figure()

for i, (rx, ry) in enumerate(resampled):
    fig.add_trace(go.Scatter(
        x=rx, y=ry,
        mode='lines', name=f'Run {i+1}',
        line=dict(color='tomato', width=1),
        opacity=0.3,
        legendgroup='runs', showlegend=(i == 0)
    ))

fig.add_trace(go.Scatter(
    x=last_plan['x'], y=last_plan['y'],
    mode='lines', name='Planned path',
    line=dict(color='royalblue', dash='dash', width=2)
))
fig.add_trace(go.Scatter(
    x=median_x, y=median_y,
    mode='lines', name='Median path',
    line=dict(color='black', width=2.5)
))

fig.update_layout(
    title='Paths + Median',
    xaxis_title='X (m)', yaxis_title='Y (m)',
    yaxis_scaleanchor='x', hovermode='closest'
)
fig.show()

### 3. **Localization Quality**